# ECG Exploration Notebook

This notebook walks through downloading, loading, filtering, detecting R-peaks, and summarizing ECG signals from the MIT-BIH Arrhythmia Database.

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((parent for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (parent / 'src').exists()), NOTEBOOK_DIR.parent)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.download_dataset import download_mitdb
from src.ecg_parser import load_annotations, load_record
from src.filters import bandpass_filter
from src.rpeak_detection import detect_r_peaks
from src.bpm_analysis import calculate_average_bpm, calculate_rr_intervals, calculate_bpm
from src.qrs_analysis import analyze_qrs
from src.visualization import plot_filtered_ecg, plot_raw_ecg, plot_r_peaks

DATASET_DIR = PROJECT_ROOT / 'data' / 'mitdb'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'figures'
RECORD_ID = '100'


## Load Record and Annotations

In [ ]:
if not (DATASET_DIR / f'{RECORD_ID}.hea').exists():
    download_mitdb(DATASET_DIR)

signal, fs, metadata = load_record(RECORD_ID, DATASET_DIR)
annotations, labels = load_annotations(RECORD_ID, DATASET_DIR)


Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

## Raw ECG

In [ ]:
plot_raw_ecg(signal, fs, OUTPUT_DIR, record_id=RECORD_ID)
time = np.arange(signal.size) / fs
plt.figure(figsize=(16, 4))
plt.plot(time, signal, linewidth=1)
plt.title(f'Raw ECG - Record {RECORD_ID}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.25)
plt.show()

## Filtered ECG

In [ ]:
filtered_signal = bandpass_filter(signal, fs)
plot_filtered_ecg(filtered_signal, fs, OUTPUT_DIR, record_id=RECORD_ID)

plt.figure(figsize=(16, 4))
plt.plot(time, filtered_signal, linewidth=1, color='tab:orange')
plt.title(f'Filtered ECG - Record {RECORD_ID}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.25)
plt.show()

## R-Peak Detection

In [ ]:
r_peaks, peak_count = detect_r_peaks(filtered_signal, fs)
plot_r_peaks(filtered_signal, r_peaks, fs, OUTPUT_DIR, record_id=RECORD_ID)

peak_count

## BPM and RR Intervals

In [ ]:
rr_intervals = calculate_rr_intervals(r_peaks, fs)
instantaneous_bpm = calculate_bpm(rr_intervals)
average_bpm = calculate_average_bpm(rr_intervals)

{
    'rr_intervals_seconds': rr_intervals[:10],
    'instantaneous_bpm': instantaneous_bpm[:10],
    'average_bpm': average_bpm,
}

## QRS Statistics

In [ ]:
qrs_metrics = analyze_qrs(filtered_signal, r_peaks, fs)
qrs_metrics

## Annotation Snapshot

In [ ]:
list(zip(annotations[:10], labels[:10]))